# Actividad 4. Selección de modelos y prototipado rápido
### Comparación de tres modelos preentrenados para lectura de placas — Unidad Habitacional TecMilenio

**Objetivo:** comparar tres modelos mediante inferencia en Google Colab, medir exactitud de placa completa, precisión, recuperación, F1, tasa de error por carácter (CER, *Character Error Rate*) y latencia. El reconocimiento óptico de caracteres (OCR, *Optical Character Recognition*) es el componente evaluado. La unidad de procesamiento gráfico (GPU, *Graphics Processing Unit*) acelera las operaciones.

**Advertencias:** La fuente Sonne contiene etiquetas automáticas que no equivalen a transcripciones verificadas. Los resultados con esas etiquetas se mostrarán únicamente como **exploratorios**. La evaluación formal requiere un archivo CSV (*Comma-Separated Values*, valores separados por comas) con etiquetas revisadas por una persona. No confundir pruebas académicas con el objetivo operativo ≥97 % de la Fase I.

**Orígenes de referencia:** [Sonne](https://huggingface.co/datasets/sonnetechnology/license-plate-text-recognition-full), [TrOCR Small](https://huggingface.co/microsoft/trocr-small-printed), [TrOCR Base placas](https://huggingface.co/DunnBC22/trocr-base-printed_license_plates_ocr), [GOT-OCR 2.0](https://huggingface.co/stepfun-ai/GOT-OCR2_0).

### Decisiones procedentes de la auditoría

En la auditoría previa se examinaron 150 imágenes de Sonne, se generaron 160 recortes y se encontraron 30 regiones de prioridad alta, 75 media y 55 baja, según filtros exploratorios. Se observaron etiquetas incompletas, caracteres inventados y recortes de resolución reducida. Una extracción inicialmente defectuosa del campo JSON de otro dataset fue corregida: la ausencia aparente de 150 etiquetas era un error del código, no del origen. **La verificación manual completa no terminó**, y se registra como deuda técnica.

La selección de los modelos considera adecuación a la tarea, requisitos de memoria, licencia declarada, facilidad de reproducción y velocidad; la clasificación final solo podrá sustentarse en mediciones sobre **las mismas imágenes**. Models, Datasets y Spaces son las secciones de modelos, datos y demostraciones interactivas del ecosistema Hugging Face. Los modelos se evaluarán sin ajuste fino (*fine-tuning*) inicial. El primer paso es un piloto pequeño y después una muestra ampliada.

In [1]:
# CELDA 1 — Inspección de GPU y entorno (activar GPU T4 desde Entorno de ejecución)
import sys, platform, subprocess
import torch
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__)
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Dispositivo:', torch.cuda.get_device_name(0))
    subprocess.run(['nvidia-smi'], check=False)
else:
    print('AVISO: habilita GPU antes de descargar los modelos; el notebook no obliga a usarla.')

Python: 3.13.15 | PyTorch: 2.11.0+cu130
GPU disponible: True
Dispositivo: Tesla T4


In [4]:

# CELDA 2 — Instalación de dependencias compatibles
# Actividad 4. Selección de modelos y prototipado rápido
#
# Objetivo:
# Resolver los conflictos de dependencias y preparar
# Google Colab para ejecutar los modelos de OCR.
#
# OCR = Optical Character Recognition
# (Reconocimiento óptico de caracteres).

# 1. Retirar gcsfs para evitar conflictos con fsspec.
%pip uninstall -y gcsfs

# 2. Instalar versiones compatibles.
%pip install -q \
    "transformers>=4.40,<5" \
    "datasets==3.6.0" \
    "fsspec[http]==2025.3.0" \
    "accelerate>=0.28,<2" \
    "huggingface-hub>=0.36,<1" \
    "pandas>=2,<3" \
    "rapidfuzz>=3,<4" \
    "pillow>=10,<13" \
    "matplotlib>=3.7,<4" \
    "sentencepiece>=0.2,<1" \
    "tiktoken>=0.6,<1"

# 3. Comprobar las dependencias.
%pip check

print("CELDA 2 — Proceso de instalación finalizado.")


Found existing installation: gcsfs 2025.12.0
Uninstalling gcsfs-2025.12.0:
  Successfully uninstalled gcsfs-2025.12.0
bigframes 2.49.0 requires gcsfs, which is not installed.
ipython 7.34.0 requires jedi, which is not installed.
diffusers 0.40.0 has requirement huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2.
gradio 6.29.0 has requirement huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2.
CELDA 2 — Proceso de instalación finalizado.


In [5]:

# CELDA 2A — Verificación funcional de dependencias
# Comprobamos únicamente las bibliotecas necesarias para el prototipo.

import importlib
import torch

bibliotecas = [
    "transformers",
    "datasets",
    "accelerate",
    "huggingface_hub",
    "pandas",
    "PIL",
    "rapidfuzz"
]

errores = []

for nombre in bibliotecas:
    try:
        modulo = importlib.import_module(nombre)
        version = getattr(modulo, "__version__", "Disponible")
        print(f"OK - {nombre}: {version}")
    except Exception as e:
        errores.append((nombre, str(e)))
        print(f"ERROR - {nombre}: {e}")

print("\nGPU disponible:", torch.cuda.is_available())

if errores:
    print("\nExisten bibliotecas que requieren corrección.")
else:
    print("\nDependencias principales verificadas correctamente.")


OK - transformers: 4.57.6
OK - datasets: 3.6.0
OK - accelerate: 1.15.0
OK - huggingface_hub: 0.36.2
OK - pandas: 2.2.3
OK - PIL: 11.3.0
OK - rapidfuzz: 3.14.6

GPU disponible: True

Dependencias principales verificadas correctamente.


In [6]:
# CELDA 3 — Importaciones, semilla y directorios
import gc, os, re, json, time, math, random, shutil, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from datasets import load_dataset
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from rapidfuzz.distance import Levenshtein
from huggingface_hub import model_info
random.seed(42); np.random.seed(42); torch.manual_seed(42)
CARPETA = Path('/content/actividad4_comparacion_ocr')
IMAGENES = CARPETA/'data'/'recortes'; RESULTADOS = CARPETA/'results'
IMAGENES.mkdir(parents=True,exist_ok=True); RESULTADOS.mkdir(parents=True,exist_ok=True)
DISPOSITIVO = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Carpeta:',CARPETA,'| dispositivo:',DISPOSITIVO)

Carpeta: /content/actividad4_comparacion_ocr | dispositivo: cuda


In [7]:
# CELDA 4 — Autenticación opcional y segura con Hugging Face Hub
# No escribas tokens directamente en el notebook ni en GitHub.
from huggingface_hub import HfApi
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token: os.environ['HF_TOKEN'] = token
except Exception:
    pass
print('Hugging Face Hub: acceso', 'autenticado' if os.getenv('HF_TOKEN') else 'público sin token')
print('Para modelos públicos no hace falta un token; la cuenta autenticada puede tener mejores límites.')

Hugging Face Hub: acceso público sin token
Para modelos públicos no hace falta un token; la cuenta autenticada puede tener mejores límites.


In [8]:
# CELDA 5 — Descarga limitada de Sonne y extracción de regiones
# Solo 150 imágenes iniciales: no asumir que los textos automáticos son correctos.
MAX_IMAGENES = 150
try:
    ds = load_dataset('sonnetechnology/license-plate-text-recognition-full', split='train', streaming=True)
    fuente = list(ds.take(MAX_IMAGENES))
    print('Imágenes recibidas:',len(fuente))
except Exception as ex:
    raise RuntimeError(f'No se pudo acceder a Sonne: {type(ex).__name__}: {ex}')
filas=[]
for indice, item in enumerate(fuente):
    foto=item['image'].convert('RGB')
    cajas=item.get('bbox') or []
    textos=item.get('target') or []
    if len(cajas)!=len(textos):
        print('Descartado por falta de correspondencia:', indice)
        continue
    for j,(c,t) in enumerate(zip(cajas,textos)):
        try:
            x1,y1,x2,y2=map(float,c)
            if not (0<=x1<x2<=foto.width and 0<=y1<y2<=foto.height): continue
            rec=foto.crop((math.floor(x1),math.floor(y1),math.ceil(x2),math.ceil(y2)))
            rid=f'sonne_{indice:05d}_region_{j}'
            rec.save(IMAGENES/f'{rid}.png')
            filas.append({'id_recorte':rid,'imagen':str(IMAGENES/f'{rid}.png'), 'etiqueta_original':str(t), 'etiqueta_verificada':'', 'estado':'sin_verificar', 'fuente':'Sonne'})
        except (ValueError,TypeError,OSError):
            continue
base=pd.DataFrame(filas)
if base.empty: raise ValueError('No se obtuvieron recortes.')
print('Recortes generados:',len(base)); display(base.head())

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Imágenes recibidas: 150
Recortes generados: 160


             id_recorte                                             imagen  \
0  sonne_00000_region_0  /content/actividad4_comparacion_ocr/data/recor...   
1  sonne_00000_region_1  /content/actividad4_comparacion_ocr/data/recor...   
2  sonne_00001_region_0  /content/actividad4_comparacion_ocr/data/recor...   
3  sonne_00002_region_0  /content/actividad4_comparacion_ocr/data/recor...   
4  sonne_00003_region_0  /content/actividad4_comparacion_ocr/data/recor...   

  etiqueta_original etiqueta_verificada         estado fuente  
0           X78IAEN                      sin_verificar  Sonne  
1              SUBT                      sin_verificar  Sonne  
2     TOTAL: 853.25                      sin_verificar  Sonne  
3               SR:                      sin_verificar  Sonne  
4        51F-826.06                      sin_verificar  Sonne  

In [9]:
# CELDA 6 — Inspección visual acotada (no equivale a verificación)
seleccion=base.sample(n=min(12,len(base)), random_state=42)
fig,ax=plt.subplots(math.ceil(len(seleccion)/4),4,figsize=(15,3*math.ceil(len(seleccion)/4)))
for a in np.asarray(ax).reshape(-1): a.axis('off')
for a,(_,r) in zip(np.asarray(ax).reshape(-1),seleccion.iterrows()):
    a.imshow(Image.open(r['imagen']));a.set_title(f"{r['id_recorte']}\n{r['etiqueta_original']}",fontsize=8)
plt.tight_layout();plt.show()
print('No calificar como correctos solo por aparecer en esta gráfica.')

<Figure size 1500x900 with 12 Axes>

No calificar como correctos solo por aparecer en esta gráfica.


### Referencias verificadas y protocolo de evaluación

Si tienes el archivo de auditoría manual `validacion_manual_placas.csv`, súbelo a la sesión de Colab. Sus campos deben ser `id_recorte`, `etiqueta_verificada`, `estado`, con `estado` igual a `validada` o `corregida`. **Las transcripciones deben proceder de revisión humana, no del resultado de los modelos comparados**. Las referencias no verificadas de Sonne se usarán únicamente para análisis exploratorio. Para evaluar con rigor recomendamos revisar al menos 30–50 registros diversos (no solo los de baja prioridad) y señalar la incertidumbre estadística de una muestra reducida.

La normalización elimina separadores y espacios tanto en referencia como en predicción para evaluar **caracteres alfanuméricos**; la imagen y la cadena original permanecen intactas. La exactitud de placa completa es más exigente que la exactitud por carácter.

In [10]:
# CELDA 7 — Importar verificación manual o preparar evaluación exploratoria
# OPCIÓN 1: subir con el control de archivos de Colab un CSV revisado.
# OPCIÓN 2: dejar la ruta vacía, usando etiquetas automáticas SOLO exploratoriamente.
RUTA_VERIFICADA = ''  # p. ej. '/content/validacion_manual_placas.csv'
if RUTA_VERIFICADA and Path(RUTA_VERIFICADA).exists():
    manual=pd.read_csv(RUTA_VERIFICADA,dtype=str,keep_default_na=False)
    requeridos={'id_recorte','etiqueta_verificada','estado'}
    if not requeridos.issubset(manual.columns): raise ValueError('CSV manual sin columnas requeridas')
    manual=manual[manual['estado'].str.lower().isin(['validada','corregida'])].drop_duplicates('id_recorte',keep='last')
    datos=base.drop(columns=['etiqueta_verificada','estado']).merge(manual[['id_recorte','etiqueta_verificada','estado']],on='id_recorte',how='inner')
    datos=datos[datos.etiqueta_verificada.str.strip().ne('')].copy()
    TIPO='FORMAL_VERIFICADA'
else:
    datos=base.copy();datos['etiqueta_verificada']=datos['etiqueta_original']
    TIPO='EXPLORATORIA_ETIQUETAS_AUTOMATICAS'
# Exclusiones obvias antes de la evaluación exploratoria; NO sustituyen revisión humana.
datos=datos[datos['etiqueta_verificada'].str.strip().ne('')].copy()
datos=datos[datos['etiqueta_verificada'].str.len().between(4,16)].copy()
datos=datos[datos['etiqueta_verificada'].str.contains(r'[A-Za-z0-9]',regex=True)].copy()
if datos.empty: raise ValueError('No hay datos para evaluar. Revisa CSV y filtros.')
# Selección reproductible, no aleatoria en cada ejecución.
MAX_PRUEBAS = 12  # piloto: ampliar a 30 o 50 después de confirmar ejecución
prueba=datos.sample(min(MAX_PRUEBAS,len(datos)),random_state=42).reset_index(drop=True)
prueba.to_csv(RESULTADOS/'muestra_evaluada.csv',index=False,encoding='utf-8-sig')
print('Modalidad:',TIPO,'| imágenes seleccionadas:',len(prueba))
if TIPO!='FORMAL_VERIFICADA': print('ADVERTENCIA: NO publicar cifras como desempeño real verificado.')
display(prueba[['id_recorte','etiqueta_verificada','estado']].head())

Modalidad: EXPLORATORIA_ETIQUETAS_AUTOMATICAS | imágenes seleccionadas: 12
ADVERTENCIA: NO publicar cifras como desempeño real verificado.


             id_recorte etiqueta_verificada         estado
0  sonne_00059_region_0               39458  sin_verificar
1  sonne_00081_region_0            OA-55-91  sin_verificar
2  sonne_00038_region_0          60A-359.81  sin_verificar
3  sonne_00050_region_0          51F-042.75  sin_verificar
4  sonne_00132_region_0          51A-853.25  sin_verificar

In [14]:

# CELDA 8 — Selección definitiva de modelos candidatos
# Actividad 4. Selección de modelos y prototipado rápido
#
# Objetivo:
# Registrar tres modelos preentrenados para comparar
# reconocimiento óptico de caracteres (OCR).
#
# Se sustituye el modelo especializado en placas debido
# al error HTTP 401 observado durante su consulta.

MODELOS = [
    {
        "nombre": "TrOCR Small impreso",
        "id": "microsoft/trocr-small-printed",
        "tipo": "trocr"
    },
    {
        "nombre": "TrOCR Base impreso",
        "id": "microsoft/trocr-base-printed",
        "tipo": "trocr"
    },
    {
        "nombre": "GOT-OCR 2.0",
        "id": "stepfun-ai/GOT-OCR2_0",
        "tipo": "got"
    }
]

metadatos = []

for modelo in MODELOS:
    try:
        info = model_info(modelo["id"], token=False)

        licencia = getattr(
            info.cardData, "license", None
        )

        metadatos.append({
            "modelo": modelo["nombre"],
            "repositorio": modelo["id"],
            "licencia_declarada": licencia or "No declarada",
            "sha_revision": info.sha,
            "estado": "consultado"
        })

    except Exception as error:
        metadatos.append({
            "modelo": modelo["nombre"],
            "repositorio": modelo["id"],
            "licencia_declarada": "Por verificar",
            "sha_revision": "",
            "estado": f"Error: {str(error)[:100]}"
        })

fichas = pd.DataFrame(metadatos)

display(fichas)

fichas.to_csv(
    RESULTADOS / "fichas_modelos.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Revisión de candidatos finalizada.")
print("Las licencias y condiciones de uso requieren revisión.")


                modelo                    repositorio licencia_declarada  \
0  TrOCR Small impreso  microsoft/trocr-small-printed       No declarada   
1   TrOCR Base impreso   microsoft/trocr-base-printed       No declarada   
2          GOT-OCR 2.0          stepfun-ai/GOT-OCR2_0         apache-2.0   

                               sha_revision      estado  
0  04e994ab854b0089d4929f48c2b4dbe2ce78a340  consultado  
1  93450be3f1ed40a930690d951ef3932687cc1892  consultado  
2  979938bf89ccdc949c0131ddd3841e24578a4742  consultado  

Revisión de candidatos finalizada.
Las licencias y condiciones de uso requieren revisión.


In [12]:

# CELDA 8A — Verificar acceso al segundo modelo
from huggingface_hub import model_info

modelo_id = "DunnBC22/trocr-base-printed_license_plates_ocr"

try:
    info = model_info(modelo_id, token=False)
    print("Modelo encontrado:", info.id)
    print("Revisión:", info.sha)
    print("Acceso público: confirmado")
except Exception as error:
    print("No se pudo consultar el modelo.")
    print("Tipo de error:", type(error).__name__)
    print("Detalle:", str(error)[:500])


No se pudo consultar el modelo.
Tipo de error: RepositoryNotFoundError
Detalle: 401 Client Error. (Request ID: Root=1-6acaa389-581fd0cd028bc3b63a872990;75f63a70-1040-43f8-a23e-fb2f322e1dd1)

Repository Not Found for url: https://huggingface.co/api/models/DunnBC22/trocr-base-printed_license_plates_ocr.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.


In [13]:

# CELDA 8B — Comprobación directa del repositorio
import requests

url = (
    "https://huggingface.co/"
    "DunnBC22/trocr-base-printed_license_plates_ocr/"
    "resolve/main/config.json"
)

try:
    respuesta = requests.get(url, timeout=15)
    print("Código HTTP:", respuesta.status_code)
    print("Archivo accesible:", respuesta.status_code == 200)
except requests.RequestException as error:
    print("Error de conexión:", str(error)[:300])


Código HTTP: 401
Archivo accesible: False


In [15]:
# CELDA 9 — Funciones de inferencia y medición
# Procesamos una imagen a la vez para evitar desbordar la GPU T4.
def normalizar(s):
    return ''.join(re.findall(r'[A-Z0-9]+',str(s).upper()))

def crear_predictor(modelo):
    proc=TrOCRProcessor.from_pretrained(modelo)
    red=VisionEncoderDecoderModel.from_pretrained(modelo).to(DISPOSITIVO).eval()
    def predecir(ruta):
        img=Image.open(ruta).convert('RGB')
        px=proc(images=img,return_tensors='pt').pixel_values.to(DISPOSITIVO)
        with torch.inference_mode():
            ids=red.generate(px,max_new_tokens=48,num_beams=1)
        return proc.batch_decode(ids,skip_special_tokens=True)[0]
    return predecir, red

def liberar(red=None):
    del red; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def tiempo_sincronizado():
    if torch.cuda.is_available(): torch.cuda.synchronize()
    return time.perf_counter()

def metrica_caracteres(verdadero,predicho):
    a,b=normalizar(verdadero),normalizar(predicho)
    if not a: return None
    operaciones=Levenshtein.editops(a,b)
    rep=sum(1 for op in operaciones if op.tag=='replace')
    ins=sum(1 for op in operaciones if op.tag=='insert')
    dele=sum(1 for op in operaciones if op.tag=='delete')
    verdaderos=len(a)-rep-dele
    precision=verdaderos/(verdaderos+rep+ins) if verdaderos+rep+ins else 0
    recall=verdaderos/(verdaderos+rep+dele) if verdaderos+rep+dele else 0
    f1=(2*precision*recall/(precision+recall)) if precision+recall else 0
    return {'acierto_exacto':int(a==b),'precision_caracteres':precision,'recall_caracteres':recall,'f1_caracteres':f1,'cer':(rep+ins+dele)/len(a),'tp':verdaderos,'substituciones':rep,'inserciones':ins,'eliminaciones':dele}
print('Funciones de inferencia y métricas preparadas.')

Funciones de inferencia y métricas preparadas.


In [16]:
# CELDA 10 — Ejecución de TrOCR Small y TrOCR Base (uno tras otro)
filas_pred=[]; estado_modelos=[]
for m in MODELOS[:2]:
    print('Modelo:',m['nombre'])
    red=None
    try:
        inicio_descarga=time.perf_counter()
        predictor,red=crear_predictor(m['id'])
        carga=time.perf_counter()-inicio_descarga
        # Calentamiento NO medido, para no confundir arranque con inferencia.
        predictor(prueba.iloc[0]['imagen'])
        for _,r in prueba.iterrows():
            t0=tiempo_sincronizado()
            pred=predictor(r['imagen'])
            lat=tiempo_sincronizado()-t0
            filas_pred.append({'modelo':m['nombre'],'id_recorte':r['id_recorte'],'real':r['etiqueta_verificada'],'prediccion':pred,'latencia_seg':lat})
        estado_modelos.append({'modelo':m['nombre'],'estado':'OK','tiempo_carga_seg':round(carga,2),'error':''})
    except Exception as e:
        estado_modelos.append({'modelo':m['nombre'],'estado':'ERROR','tiempo_carga_seg':None,'error':f'{type(e).__name__}: {str(e)[:350]}'})
        print('FALLO DOCUMENTADO:',estado_modelos[-1]['error'])
    finally:
        if red is not None: red.cpu()
        liberar(red)
pd.DataFrame(estado_modelos).to_csv(RESULTADOS/'estado_modelos_parcial.csv',index=False)
print('Predicciones calculadas:',len(filas_pred));display(pd.DataFrame(estado_modelos))

Modelo: TrOCR Small impreso


preprocessor_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Predicciones calculadas: 24


                modelo estado  tiempo_carga_seg error
0  TrOCR Small impreso     OK             12.52      
1   TrOCR Base impreso     OK             13.54      

### GOT-OCR 2.0: ejecución condicionada

La ficha original de GOT-OCR 2.0 incluye dependencias antiguas y ejecución con código remoto de terceros. Su compatibilidad con el Colab actual no está garantizada. La CELDA 11 **no ejecuta código remoto sin tu autorización**: se controla mediante `ACEPTO_CODIGO_REMOTO`. Si decides no habilitarlo, quedará documentado como candidato no probado y **la rúbrica de tres modelos no estará cubierta**; habrá que sustituirlo por un tercero compatible. Revisa previamente el repositorio y confía únicamente en código que hayas evaluado.

In [18]:

# ============================================================
# CELDA 11 — Ejecución del tercer modelo OCR
# Actividad 4. Selección de modelos y prototipado rápido
# ============================================================
#
# OCR = Optical Character Recognition
# (Reconocimiento óptico de caracteres).
#
# Se sustituye GOT-OCR 2.0 por TrOCR Small Handwritten.
# Motivo: evitar ejecución de código remoto no revisado.
#
# LIMITACIÓN:
# Este modelo fue entrenado para texto manuscrito.
# Su evaluación sobre placas impresas permite analizar
# el efecto de utilizar un modelo de otro dominio.
# ============================================================

modelo_tercero = {
    "nombre": "TrOCR Small manuscrito",
    "id": "microsoft/trocr-small-handwritten",
    "tipo": "trocr"
}

# Actualizar la lista de modelos candidatos.
MODELOS[2] = modelo_tercero

# Eliminar un resultado anterior no ejecutado de GOT-OCR.
estado_modelos = [
    e for e in estado_modelos
    if e["modelo"] != "GOT-OCR 2.0"
]

red = None

print("Modelo:", modelo_tercero["nombre"])

inicio_carga = time.perf_counter()

try:

    predictor, red = crear_predictor(
        modelo_tercero["id"]
    )

    tiempo_carga = time.perf_counter() - inicio_carga

    # Calentamiento de GPU.
    predictor(prueba.iloc[0]["imagen"])

    for _, registro in prueba.iterrows():

        inicio = tiempo_sincronizado()

        prediccion = predictor(
            registro["imagen"]
        )

        latencia = tiempo_sincronizado() - inicio

        filas_pred.append({
            "modelo": modelo_tercero["nombre"],
            "id_recorte": registro["id_recorte"],
            "real": registro["etiqueta_verificada"],
            "prediccion": prediccion,
            "latencia_seg": latencia
        })

    estado_modelos.append({
        "modelo": modelo_tercero["nombre"],
        "estado": "OK",
        "tiempo_carga_seg": round(tiempo_carga, 2),
        "error": ""
    })

except Exception as error:

    estado_modelos.append({
        "modelo": modelo_tercero["nombre"],
        "estado": "ERROR",
        "tiempo_carga_seg": None,
        "error": f"{type(error).__name__}: {str(error)[:350]}"
    })

    print("Error:", type(error).__name__, str(error)[:350])

finally:

    if red is not None:
        red.cpu()

    liberar(red)

# Guardar resultados actualizados.
df_estados = pd.DataFrame(estado_modelos)

df_estados.to_csv(
    RESULTADOS / "estado_modelos.csv",
    index=False,
    encoding="utf-8-sig"
)

display(df_estados)

print("Predicciones acumuladas:", len(filas_pred))


Modelo: TrOCR Small manuscrito


preprocessor_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

...

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-small-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/246M [00:00<?, ?B/s]

                   modelo estado  tiempo_carga_seg error
0     TrOCR Small impreso     OK             12.52      
1      TrOCR Base impreso     OK             13.54      
2  TrOCR Small manuscrito     OK              5.90      

Predicciones acumuladas: 36


In [19]:
# CELDA 12 — Métricas calculadas sobre las mismas imágenes; sin rellenar resultados faltantes
predicciones=pd.DataFrame(filas_pred)
if predicciones.empty: raise ValueError('No hay predicciones: revisar errores anteriores.')
metricas=[]
for _,r in predicciones.iterrows():
    m=metrica_caracteres(r['real'],r['prediccion'])
    if m: metricas.append({**r.to_dict(),**m})
detalle=pd.DataFrame(metricas)
resumen=detalle.groupby('modelo').agg(
    muestras=('id_recorte','nunique'),accuracy_placa=('acierto_exacto','mean'),
    precision=('precision_caracteres','mean'), recall=('recall_caracteres','mean'),
    f1=('f1_caracteres','mean'), cer=('cer','mean'),
    latencia_media_s=('latencia_seg','mean'), latencia_mediana_s=('latencia_seg','median')
).reset_index()
# Métricas promedio por matrícula (macro) y no agregadas por carácter (micro).
conjuntos={m:set(g.id_recorte) for m,g in detalle.groupby('modelo')}
mismo_conjunto=(len(conjuntos)==3 and len({tuple(sorted(x)) for x in conjuntos.values()})==1)
print('Modalidad:',TIPO, '| comparables los tres:',mismo_conjunto)
if not mismo_conjunto: print('ADVERTENCIA: NO existe una comparación completa de tres modelos; no seleccionar ganador.')
display(resumen.round(4))
detalle.to_csv(RESULTADOS/'predicciones_detalle.csv',index=False,encoding='utf-8-sig')
resumen.to_csv(RESULTADOS/'metricas_resumen.csv',index=False,encoding='utf-8-sig')

Modalidad: EXPLORATORIA_ETIQUETAS_AUTOMATICAS | comparables los tres: True


                   modelo  muestras  accuracy_placa  precision  recall  \
0      TrOCR Base impreso        12          0.6667     0.8496  0.8792   
1     TrOCR Small impreso        12          0.5000     0.7333  0.6806   
2  TrOCR Small manuscrito        12          0.0000     0.0168  0.0208   

       f1     cer  latencia_media_s  latencia_mediana_s  
0  0.8526  0.2250            0.1137              0.1224  
1  0.6955  0.3403            0.0562              0.0557  
2  0.0186  1.0521            0.0441              0.0420  

In [20]:
# CELDA 13 — Evidencias de errores y comparación de latencias
if not detalle.empty:
    peores=detalle.sort_values('cer',ascending=False)[['modelo','id_recorte','real','prediccion','cer']].head(12)
    display(peores)
    fig,ax=plt.subplots(figsize=(8,4))
    ax.bar(resumen['modelo'],resumen['latencia_media_s'])
    ax.set_ylabel('Segundos por imagen (media)');ax.set_title('Latencia en misma sesión de Colab')
    plt.xticks(rotation=20,ha='right');plt.tight_layout();plt.show()
print('Los resultados en una muestra pequeña requieren cautela y no representan disponibilidad 24/7 ni tiempo de apertura física.')

                    modelo            id_recorte        real    prediccion  \
27  TrOCR Small manuscrito  sonne_00050_region_0  51F-042.75    strongness   
30  TrOCR Small manuscrito  sonne_00143_region_0  51F-574.93  55th century   
28  TrOCR Small manuscrito  sonne_00132_region_0  51A-853.25     mmeezably   
7      TrOCR Small impreso  sonne_00066_region_0       -KEEP         USHIP   
25  TrOCR Small manuscrito  sonne_00081_region_0    OA-55-91          just   
24  TrOCR Small manuscrito  sonne_00059_region_0       39458         which   
29  TrOCR Small manuscrito  sonne_00089_region_0      3.2.88          will   
32  TrOCR Small manuscrito  sonne_00048_region_0     2.59:17         which   
35  TrOCR Small manuscrito  sonne_00033_region_0   51A-01204     israorzoy   
10     TrOCR Small impreso  sonne_00094_region_3       TOTAL           SEM   
19      TrOCR Base impreso  sonne_00066_region_0       -KEEP       DESEIPT   
31  TrOCR Small manuscrito  sonne_00066_region_0       -KEEP    

<Figure size 800x400 with 1 Axes>

Los resultados en una muestra pequeña requieren cautela y no representan disponibilidad 24/7 ni tiempo de apertura física.


In [21]:
# CELDA 14 — Conclusión controlada y paquete reproducible de resultados
status=pd.DataFrame(estado_modelos)
status.to_csv(RESULTADOS/'estado_modelos.csv',index=False,encoding='utf-8-sig')
texto=['# Resultados y decisión técnica — Actividad 4','',f'- Modalidad de evaluación: **{TIPO}**.',f'- Registros evaluados: **{len(prueba)}**.',f'- Modelos ejecutados correctamente: **{sum(status.estado.eq("OK"))} de 3**.','',
'- Advertencia: las etiquetas de Sonne no son automáticamente una verdad de referencia; sin validación manual, estas métricas son exploratorias.',
'- No se infiere cumplimiento del 97 % de lectura ni se habilita control de acceso físico.','',
'## Deuda técnica','- Verificar manualmente matrículas representativas y su transcripción.','- Asegurar ausencia de solapamiento con los conjuntos de entrenamiento.','- Medir memoria máxima de GPU y gasto operativo en escenarios realistas.','- Comprobar formatos mexicanos y dos líneas, incluyendo problemas de iluminación.','- Finalizar validación de las tres implementaciones en el mismo entorno.']
if mismo_conjunto and TIPO=='FORMAL_VERIFICADA':
    ganador=resumen.sort_values(['f1','latencia_media_s'],ascending=[False,True]).iloc[0]['modelo']
    texto += ['',f'## Modelo con mayor F1 en esta muestra: {ganador}', 'La recomendación es provisional y debe contrastarse con latencia, licencia, memoria y requisitos del negocio.']
else:
    texto += ['', '## Selección final pendiente', 'Se requiere ejecutar los tres modelos y verificar referencias para justificar la selección con métricas comparables.']
(RESULTADOS/'conclusiones_preliminares.md').write_text('\n'.join(texto),encoding='utf-8')
archivo=shutil.make_archive('/content/actividad4_resultados','zip',root_dir=RESULTADOS)
print('Resultados:',RESULTADOS,'| paquete opcional de respaldo:',archivo)
print('GitHub: publicar notebook, CSV pequeños, README y conclusiones; no publicar tokens ni imágenes sin revisar permisos.')

Resultados: /content/actividad4_comparacion_ocr/results | paquete opcional de respaldo: /content/actividad4_resultados.zip
GitHub: publicar notebook, CSV pequeños, README y conclusiones; no publicar tokens ni imágenes sin revisar permisos.


### Lista de cierre de la rúbrica

- **25 puntos — Selección:** se describieron Models, Datasets y Spaces; deben confirmarse licencias, versiones y tres modelos aptos.
- **25 puntos — Implementación:** GPU habilitada, dependencias documentadas, notebook por celdas; los tres estados deben ser `OK`.
- **25 puntos — Evaluación:** accuracy de placa completa, precision, recall, F1, CER y latencia calculados sobre exactamente las mismas referencias verificadas.
- **25 puntos — Documentación:** subir a GitHub el notebook, `requirements.txt`, archivos CSV, análisis y dictamen; explicar los errores y compromisos de precisión frente a eficiencia.

**Pendientes genuinos:** comprobar comportamiento real en Colab, revisar código remoto GOT antes de usarlo, completar muestra de etiquetas humanas y publicar repositorio. No reportar calificación de 100 % hasta cubrir esos puntos.